In [7]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import os

df = pd.read_csv('../data/processed/cleaned.csv')
print("Shape:", df.shape)
df.head()

Shape: (2927, 78)


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Lot Shape,Land Contour,Utilities,...,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,IR1,Lvl,AllPub,...,0,0,0,0,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,Reg,Lvl,AllPub,...,0,0,120,0,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,IR1,Lvl,AllPub,...,0,0,0,0,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,Reg,Lvl,AllPub,...,0,0,0,0,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,IR1,Lvl,AllPub,...,0,0,0,0,0,3,2010,WD,Normal,189900


In [8]:
def generate_doc(row):
    price = int(row['SalePrice'])
    
    quality_labels = {
        'Ex': 'Excellent', 'Gd': 'Good', 
        'TA': 'Average', 'Fa': 'Fair', 'Po': 'Poor'
    }
    kitchen = quality_labels.get(row['Kitchen Qual'], row['Kitchen Qual'])
    
    return f"""
    A {int(row['Gr Liv Area'])} sq ft home in the {row['Neighborhood'].strip()} 
    neighborhood sold for ${price:,}. The property has an Overall Quality 
    rating of {int(row['Overall Qual'])} out of 10, built in {int(row['Year Built'])}. 
    It features {int(row['Full Bath'])} full bathrooms, {int(row['Bedroom AbvGr'])} 
    bedrooms, and a {int(row['Garage Cars'])}-car garage. 
    The kitchen quality is {kitchen} and the 
    basement size is {int(row['Total Bsmt SF'])} sq ft.
    """.strip()

documents = df.apply(generate_doc, axis=1).tolist()

print(f"Generated {len(documents)} documents")
print("\nSample document:")
print(documents[0])

Generated 2927 documents

Sample document:
A 1656 sq ft home in the NAmes 
    neighborhood sold for $215,000. The property has an Overall Quality 
    rating of 6 out of 10, built in 1960. 
    It features 1 full bathrooms, 3 
    bedrooms, and a 2-car garage. 
    The kitchen quality is Average and the 
    basement size is 1080 sq ft.


In [9]:
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding documents... (this may take 2-3 minutes)")
embeddings = embedder.encode(documents, show_progress_bar=True)

print(f"\nEmbedding shape: {embeddings.shape}")
# Should be (2927, 384) — 2927 documents, 384 dimensions per embedding

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding documents... (this may take 2-3 minutes)


Batches:   0%|          | 0/92 [00:00<?, ?it/s]


Embedding shape: (2927, 384)


In [10]:
dimension = embeddings.shape[1]  # 384

index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))

print(f"FAISS index built")
print(f"Total vectors in index: {index.ntotal}")

FAISS index built
Total vectors in index: 2927


In [11]:
# Test query — describe a house and find similar ones
test_query = "3 bedroom house in CollgCr neighborhood with good kitchen quality and 2 car garage"

query_embedding = embedder.encode([test_query])
distances, indices = index.search(query_embedding.astype('float32'), k=5)

print("Top 5 most similar houses to query:")
print(f"Query: {test_query}\n")
for i, idx in enumerate(indices[0]):
    print(f"--- Match {i+1} (distance: {distances[0][i]:.4f}) ---")
    print(documents[idx])
    print()

Top 5 most similar houses to query:
Query: 3 bedroom house in CollgCr neighborhood with good kitchen quality and 2 car garage

--- Match 1 (distance: 0.5721) ---
A 1640 sq ft home in the CollgCr 
    neighborhood sold for $183,000. The property has an Overall Quality 
    rating of 7 out of 10, built in 2003. 
    It features 2 full bathrooms, 3 
    bedrooms, and a 2-car garage. 
    The kitchen quality is Good and the 
    basement size is 798 sq ft.

--- Match 2 (distance: 0.5735) ---
A 1651 sq ft home in the CollgCr 
    neighborhood sold for $246,500. The property has an Overall Quality 
    rating of 8 out of 10, built in 2007. 
    It features 2 full bathrooms, 3 
    bedrooms, and a 3-car garage. 
    The kitchen quality is Good and the 
    basement size is 1643 sq ft.

--- Match 3 (distance: 0.5771) ---
A 1459 sq ft home in the CollgCr 
    neighborhood sold for $192,000. The property has an Overall Quality 
    rating of 7 out of 10, built in 2002. 
    It features 2 full ba

In [12]:
os.makedirs('../data/vector_store', exist_ok=True)

# Save FAISS index
faiss.write_index(index, '../data/vector_store/houses.index')

# Save documents and embeddings for reference
with open('../data/vector_store/documents.pkl', 'wb') as f:
    pickle.dump(documents, f)

with open('../data/vector_store/embeddings.pkl', 'wb') as f:
    pickle.dump(embeddings, f)

print("Vector store saved")

Vector store saved
